In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

TARGET = "CKD (1-yes, 0-no)"

DATA_PATH = "DiabeticCKD_dataset.csv"

df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]


In [2]:
# Structural integrity checks
print("Raw shape:", df.shape)
print("Unique patients:", df["Patient ID"].nunique())
print("Missing values total:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())


Raw shape: (4000, 26)
Unique patients: 400
Missing values total: 0
Duplicate rows: 0


In [3]:
# Collapse to the latest observation per patient
latest_idx = df.groupby("Patient ID")["Diabetic Year"].idxmax()
data = df.loc[latest_idx].reset_index(drop=True)

print("\nAfter collapse:", data.shape)
print("Class balance:")
print(data[TARGET].value_counts(normalize=True))

assert data["Patient ID"].is_unique, "Collapsing failed"
assert data.isnull().sum().sum() == 0, "Unexpected missing values"



After collapse: (400, 26)
Class balance:
CKD (1-yes, 0-no)
0    0.5375
1    0.4625
Name: proportion, dtype: float64


In [4]:

# Engineer the five composite features
data["Weight_Change_From_Avg"] = (
    data["Weight (kg)"] - data["Average Weight (kg)"]
)
data["Comorbidity_Count"] = data[[
    "Hypertension (1-yes, 0-no)",
    "Heart Disease (1-yes, 0-no)",
    "Urinary Infection (1-yes, 0-no)"
]].sum(axis=1)
data["Poor_Lifestyle_Score"] = (
      (1 - data["Follow suggested Diet (1-yes, 0-no)"])
    +      data["Smoke (1-yes, 0-no)"]
    + (1 - data["Walk Regularly (1-yes, 0-no)"])
    + (1 - data["Sleep (1-sufficient, 0-insufficient)"])
    + (1 - data["Water Consumption (1-sufficient, 0-insufficient)"])
)
data["Diabetic_Duration_x_Insulin"] = (
    data["Diabetic Year"] * data["Take Insulin (1-yes, 0-no)"]
)
data["BMI_x_Hypertension"] = (
    data["BMI"] * data["Hypertension (1-yes, 0-no)"]
)


In [5]:
# Remove identifier (leakage guard) and encode gender
data = data.drop(columns=["Patient ID"])

le_gender = LabelEncoder()
data["Gender (M-male, F-female)"] = le_gender.fit_transform(
    data["Gender (M-male, F-female)"]
)   # F = 0, M = 1

In [6]:
# Final verification and save
print("\nFinal shape:", data.shape)
print("Missing values:", data.isnull().sum().sum())
assert "Patient ID" not in data.columns

data.to_csv("CKD_preprocessed_patient_level.csv", index=False)
print("Saved: CKD_preprocessed_patient_level.csv")



Final shape: (400, 30)
Missing values: 0
Saved: CKD_preprocessed_patient_level.csv


In [7]:
# DESCRIPTIVE PROFILE OF THE STUDY POPULATION (N = 400)
print("\n" + "=" * 64)
print("TABLE 4: DESCRIPTIVE PROFILE (N = 400)")
print("=" * 64)
print(f"Patients                     : {len(data)}")
print(f"CKD prevalence               : {int(data[TARGET].sum())} "
      f"({data[TARGET].mean()*100:.2f}%)")

n_male = int((data["Gender (M-male, F-female)"] == 1).sum())
print(f"Gender - Male                : {n_male} ({n_male/len(data)*100:.2f}%)")
print(f"Gender - Female              : {len(data)-n_male} "
      f"({(len(data)-n_male)/len(data)*100:.2f}%)")

continuous = ["Age", "Average Age", "Height (cm)", "Weight (kg)",
              "Average Weight (kg)", "BMI", "Diabetic Year",
              "Calorie Intake (per day)", "Weight_Change_From_Avg",
              "Comorbidity_Count", "Poor_Lifestyle_Score",
              "Diabetic_Duration_x_Insulin", "BMI_x_Hypertension"]
print("\n Continuous variables: mean (SD) [min, max] ")
for c in continuous:
    print(f"{c:30s}: {data[c].mean():8.2f} ({data[c].std():6.2f}) "
          f"[{data[c].min():.2f}, {data[c].max():.2f}]")

binaries = ["Family Background of Diabetes (1-yes, 0-no)",
            "Follow suggested Diet (1-yes, 0-no)",
            "Take Medicine for Diabetes (1-yes, 0-no)",
            "Take Insulin (1-yes, 0-no)",
            "Hypertension (1-yes, 0-no)",
            "Heart Disease (1-yes, 0-no)",
            "Sleep (1-sufficient, 0-insufficient)",
            "Water Consumption (1-sufficient, 0-insufficient)",
            "Smoke (1-yes, 0-no)",
            "Zarda, Betel Leaf (1-yes, 0-no)",
            "Walk Regularly (1-yes, 0-no)",
            "Urination Properly (1-yes, 0-no)",
            "Urinary Infection (1-yes, 0-no)",
            "Pain killer (1-yes, 0-no)"]
print("\n Binary variables: n (%) ")
for c in binaries:
    print(f"{c:52s}: {int(data[c].sum()):3d} ({data[c].mean()*100:5.2f}%)")

job = data["Job (1-normal, 2-intermediate, 3-heavy)"].value_counts().sort_index()
print(f"\nOccupational workload (normal / intermediate / heavy): "
      f"{job.get(1,0)} / {job.get(2,0)} / {job.get(3,0)}")


TABLE 4: DESCRIPTIVE PROFILE (N = 400)
Patients                     : 400
CKD prevalence               : 185 (46.25%)
Gender - Male                : 145 (36.25%)
Gender - Female              : 255 (63.75%)

 Continuous variables: mean (SD) [min, max] 
Age                           :    55.37 ( 10.58) [25.00, 89.00]
Average Age                   :    51.37 ( 10.58) [21.00, 85.00]
Height (cm)                   :   158.30 (  7.97) [132.08, 182.88]
Weight (kg)                   :    61.80 (  9.90) [30.00, 92.00]
Average Weight (kg)           :    61.53 (  9.76) [31.50, 90.00]
BMI                           :    24.59 (  3.78) [13.56, 41.66]
Diabetic Year                 :    13.77 (  5.82) [10.00, 41.00]
Calorie Intake (per day)      :  1453.70 (211.79) [1000.00, 2400.00]
Weight_Change_From_Avg        :     0.27 (  1.84) [-8.10, 13.20]
Comorbidity_Count             :     0.83 (  0.80) [0.00, 3.00]
Poor_Lifestyle_Score          :     1.28 (  1.06) [0.00, 5.00]
Diabetic_Duration_x_Insulin   